# Step 0 · [4] Classifier on the embeddings

`05_semantic_filter` compares each abstract with **one** description of smart glasses. On the labelled families that score barely separates smart glasses from the rest (AUC of `margin` 0.56, of `sim_definition` 0.62, of `bm25` 0.72). Here the model learns what smart glasses are from the **labelled examples** instead: a logistic regression on the Patent SBERT vectors, plus the three scores of `05` as extra features. A first test on 201 labelled families gave a cross-validated AUC of 0.83–0.85.

1. **Labels.** Every worksheet of `data/labels/`, in either format, through `src/labels.py`: our labelling rounds and the validation samples of the team's `trial_branch` query. The target is binary, as in the team worksheets: `relevance` 1 = smart glasses, 0 = not in the sector (the adjacent HMD / VR market is a 0 with a reason).
2. **Cross-validation.** Each labelled family is predicted by a model that has not seen it. We report the AUC against the three scores of `05`, and precision / recall / F1 by threshold.
3. **Scores for every family.** The model, fitted on all labels, gives each candidate family `p_sector`. A to-review family is accepted when `p_sector` reaches the threshold: **final sector = rule A + accepted to-review families**.

`p_sector` is a score between 0 and 1, not a calibrated probability (the classes are re-weighted to be balanced).

This notebook needs no PATSTAT access. **Before running:** `05_semantic_filter` with the same `ENV` (scores and embeddings), and the labelled files in `project/data/labels/`.

In [ ]:
%pip install -q scikit-learn

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.labels import load_labels

ENV = "PROD"                                    # same environment as 05_semantic_filter
OUTPUT_DIR = Path("outputs/step0")
EMB_DIR = OUTPUT_DIR / f"embeddings_{ENV}"
LABELS_DIR = Path("data/labels")
SCORE_COLS = ["sim_definition", "margin", "bm25"]
C_GRID = (0.03, 0.1, 0.3, 1.0)                  # regularisation strengths tried (smaller = stronger)
THRESHOLD = None                                # None = the threshold with the best F1 in cross-validation; or a number in (0, 1)
SEED = 42
pd.set_option("display.max_colwidth", 110)


def read_table(stem, columns=None):
    """Read outputs/step0/<stem>_<ENV> as parquet, or as CSV if parquet was not available."""
    path = OUTPUT_DIR / f"{stem}_{ENV}.parquet"
    if path.exists():
        return pd.read_parquet(path, columns=columns)
    return pd.read_csv(path.with_suffix(".csv"), usecols=columns)


scores = read_table("scores")
labels = load_labels(LABELS_DIR)
print(f"env {ENV} | {len(scores):,} scored families | {len(labels):,} families in the label files")

## 1. The labels

Category **1** smart glasses, **2** adjacent HMD / VR market (a 0 with that reason), **0** not related. Families on which two files disagree are conflicts: left out, to be decided by hand. Labelled families without scores are outside our candidate pool or have no English abstract: the labelled 1 among them are smart-glasses patents this pipeline cannot reach (see `06_external_labels` for the recall).

In [ ]:
print(pd.crosstab(labels["files"], labels["category"].fillna("unsure"), margins=True).to_string())
conflicts = labels[labels["conflict"]]
print(f"\nconflicts between files (left out): {len(conflicts)}")
if len(conflicts):
    print(conflicts[["docdb_family_id", "reviewers", "files"]].to_string(index=False))

data = (labels.dropna(subset=["relevance"])
        .merge(scores[["docdb_family_id", "tier", *SCORE_COLS]], on="docdb_family_id", how="left"))
no_score = data[data["tier"].isna()]
print(f"\nlabelled 1 / 0: {len(data):,} | without scores: {len(no_score):,}, of which labelled 1: {int((no_score['relevance'] == 1).sum()):,}")
data = data.dropna(subset=["tier"]).reset_index(drop=True)
print()
print(pd.crosstab(data["tier"], data["category"], margins=True).to_string())

## 2. Features

The Patent SBERT vector of each family (stored by `05_semantic_filter`, read part by part) and the three scores of `05`.

In [ ]:
def stored_parts():
    return sorted(EMB_DIR.glob("part_*_ids.npy"))


def vectors_path(ids_path):
    return str(ids_path).replace("_ids.npy", "_vec.npy")


def embeddings_of(family_ids):
    """Vectors of the given families, read part by part: the whole store does not have to fit in memory."""
    wanted = np.asarray(list(family_ids), dtype=np.int64)
    found = {}
    for path in stored_parts():
        ids = np.load(path)
        keep = np.isin(ids, wanted)
        if keep.any():
            found.update(zip(ids[keep].tolist(), np.load(vectors_path(path))[keep]))
    return found


vectors = embeddings_of(data["docdb_family_id"])
data = data[data["docdb_family_id"].isin(vectors)].reset_index(drop=True)
X = np.hstack([np.vstack([vectors[i] for i in data["docdb_family_id"]]),
               data[SCORE_COLS].to_numpy(dtype=float)])
y = data["relevance"].astype(int).to_numpy()
print(f"{len(y)} labelled families with a vector: {int(y.sum())} labelled 1, {int((y == 0).sum())} labelled 0 | {X.shape[1]} features")
if min(y.sum(), (y == 0).sum()) < 10:
    raise RuntimeError("Too few labels of one class to cross-validate: label more families first.")

## 3. Cross-validation

Five folds: every family is predicted by a model fitted on the other four fifths. **AUC** = probability that a random smart-glasses family scores higher than a random other one (0.5 is chance, 1 is perfect).

The labelled families are **not a random sample** of the pool (they were drawn by score band and by query block), so precision and recall below describe the labelled set; they rank the thresholds, they are not yet population estimates.

In [ ]:
def model(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000, class_weight="balanced"))


folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = {C: cross_val_predict(model(C), X, y, cv=folds, method="predict_proba")[:, 1] for C in C_GRID}
auc = pd.Series({C: roc_auc_score(y, p) for C, p in cv_scores.items()}, name="AUC")
best_C = auc.idxmax()
data["p_cv"] = cv_scores[best_C]

print("classifier, by regularisation strength C:")
print(auc.round(3).to_string())
print(f"-> C = {best_C}")
print("\nfor comparison, the scores of 05 alone:")
print(pd.Series({c: roc_auc_score(y, data[c]) for c in SCORE_COLS}, name="AUC").round(3).to_string())

review_rows = data[data["tier"] == "to review"]
if review_rows["relevance"].nunique() == 2:
    print(f"\nto-review families only (where the classifier decides), n = {len(review_rows)}:")
    print(pd.Series({"classifier": roc_auc_score(review_rows["relevance"], review_rows["p_cv"]),
                     **{c: roc_auc_score(review_rows["relevance"], review_rows[c]) for c in SCORE_COLS}},
                    name="AUC").round(3).to_string())

In [ ]:
# Precision / recall / F1 by threshold, on the to-review families when there are enough of them
evaluated = review_rows if (review_rows["relevance"].nunique() == 2 and len(review_rows) >= 40) else data
truth = evaluated["relevance"] == 1
rows = []
for t in np.round(np.arange(0.10, 0.91, 0.05), 2):
    accepted = evaluated["p_cv"] >= t
    tp = int((accepted & truth).sum())
    precision = tp / accepted.sum() if accepted.sum() else np.nan
    recall = tp / truth.sum()
    f1 = 2 * precision * recall / (precision + recall) if tp else 0.0
    rows.append({"threshold": t, "accepted": int(accepted.sum()), "precision": precision, "recall": recall, "F1": f1})
by_threshold = pd.DataFrame(rows).set_index("threshold")
threshold = float(by_threshold["F1"].idxmax()) if THRESHOLD is None else float(THRESHOLD)

print(f"{len(evaluated)} families evaluated ({'to review' if evaluated is review_rows else 'all labelled'}), {int(truth.sum())} labelled 1")
print(by_threshold.round(3).to_string())
print(f"\n-> threshold = {threshold}" + (" (best F1)" if THRESHOLD is None else " (set by hand)"))

### Where the model and the labels disagree

The labelled 1 with the lowest cross-validated score and the labelled 0 with the highest. Worth reading: they are either hard cases, or labels to check against `LABELING_GUIDE.md`.

In [ ]:
titles = read_table("texts", columns=["docdb_family_id", "appln_title"])
shown = data.merge(titles, on="docdb_family_id", how="left")
columns = ["p_cv", "category", "tier", "files", "appln_title"]
print("--- labelled 1, lowest scores")
print(shown[shown["relevance"] == 1].nsmallest(12, "p_cv")[columns].round(3).to_string(index=False))
print("\n--- labelled 0, highest scores")
print(shown[shown["relevance"] == 0].nlargest(12, "p_cv")[columns].round(3).to_string(index=False))

## 4. Score every family

The model is fitted on all the labels and applied to every family with a vector. Rule A families are scored too, for information: they stay in the sector whatever their score (rule A is about 77% precise on the labelled samples; whether to filter it as well is a decision for the group). To-review families without an English abstract have no score and cannot be accepted.

In [ ]:
final_model = model(best_C).fit(X, y)

features = scores.set_index("docdb_family_id")[SCORE_COLS]
features = features[~features.index.duplicated()]
parts = []
for path in stored_parts():
    ids = np.load(path)
    keep = np.isin(ids, features.index.to_numpy())
    if not keep.any():
        continue
    part = np.hstack([np.load(vectors_path(path))[keep], features.loc[ids[keep]].to_numpy(dtype=float)])
    parts.append(pd.DataFrame({"docdb_family_id": ids[keep], "p_sector": final_model.predict_proba(part)[:, 1]}))
predicted = pd.concat(parts, ignore_index=True).drop_duplicates("docdb_family_id")

out = (scores.merge(predicted, on="docdb_family_id", how="left")
       .merge(labels[["docdb_family_id", "category"]].rename(columns={"category": "label"}), on="docdb_family_id", how="left"))
out["accepted"] = (out["tier"] == "to review") & (out["p_sector"] >= threshold)
out["in_final_sector"] = (out["tier"] == "in sector") | out["accepted"]

rule_a = out["tier"] == "in sector"
review = out["tier"] == "to review"
summary = pd.Series({
    "in sector (rule A)": int(rule_a.sum()),
    f"  of which p_sector < {threshold} (kept, to discuss)": int((rule_a & (out["p_sector"] < threshold)).sum()),
    "to review, scored": int(review.sum()),
    f"  accepted (p_sector >= {threshold})": int(out["accepted"].sum()),
    "final sector": int(out["in_final_sector"].sum()),
}, name="families")
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for tier, grp in out.groupby("tier"):
    axes[0].hist(grp["p_sector"].dropna(), bins=50, density=True, alpha=0.5, label=tier)
axes[0].axvline(threshold, color="grey", lw=0.8)
axes[0].set(title="p_sector by tier", xlabel="p_sector", ylabel="density")
axes[0].legend()

per_year = pd.DataFrame({
    "rule A": out[rule_a].groupby("year").size(),
    "rule A + accepted": out[out["in_final_sector"]].groupby("year").size(),
}).fillna(0)
per_year.plot(ax=axes[1], marker="o", ms=3)
axes[1].set(title=f"families per earliest priority year ({ENV})", xlabel="earliest priority year", ylabel="families")
plt.tight_layout()
plt.show()

## 5. Eyeballing the to-review families

Highest scores, around the threshold, lowest scores.

In [ ]:
ranked = out[review & out["p_sector"].notna()].merge(titles, on="docdb_family_id", how="left").sort_values("p_sector", ascending=False)
near = ranked.iloc[(ranked["p_sector"] - threshold).abs().argsort()[:12]] if len(ranked) else ranked
for label, part in (("highest p_sector", ranked.head(12)), (f"around the threshold ({threshold})", near), ("lowest p_sector", ranked.tail(12))):
    print(f"--- {label}")
    print(part[["p_sector", "closest_negative", "appln_title"]].round(3).to_string(index=False))

## 6. Save

- `classified_<ENV>`: one row per scored family, with `p_sector`, `accepted` and `in_final_sector`;
- `classifier_cv_<ENV>.csv`: the labelled families with their cross-validated score, to review the disagreements.

Both stay on TIP (`outputs/` is git-ignored).

In [ ]:
columns = ["docdb_family_id", "year", "tier", "p_sector", "accepted", "in_final_sector", "label", *SCORE_COLS, "closest_negative"]
path = OUTPUT_DIR / f"classified_{ENV}.parquet"
try:
    out[columns].to_parquet(path, index=False)
except ImportError:                       # no pyarrow: fall back to CSV
    path = path.with_suffix(".csv")
    out[columns].to_csv(path, index=False)
shown[["docdb_family_id", "category", "tier", "files", "reviewers", "p_cv", "appln_title"]].to_csv(
    OUTPUT_DIR / f"classifier_cv_{ENV}.csv", index=False, encoding="utf-8-sig")
print(f"saved {len(out):,} families to {path} (C = {best_C}, threshold = {threshold}, {len(y)} labels)")

## 7. Experiment: filter Rule A too (diagnostic only)

Compare both policies on the **same scored labelled families**, using out-of-fold
predictions, at the same thresholds. The existing final-sector decision is unchanged.
`rule_a_plus_classifier` keeps Rule A automatically; `classifier_all` filters both tiers.
The comparison reports confusion counts, precision, recall and F1, and identifies the
labelled positives lost and negatives removed by filtering Rule A.

These are exploratory sample metrics: C was selected on these folds, the labels are
not a random population sample, and missing-text/unscored families are excluded.
Do not interpret the best row as an independent test result or population precision.
Full-pool counts below use the fitted model and describe volume only, not accuracy.

Rollback checkpoint: `checkpoint/goksun-before-rule-a-measurement-2026-11-12`
(commit `f723d7f`). This experiment lives on `experiment/rule-a-measurement`.


In [ ]:
# Diagnostic comparison only: never overwrite in_final_sector or threshold.
from src.policy_metrics import binary_metrics

comparison_thresholds = sorted(set([0.30, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, float(threshold)]))
is_core = data["tier"].eq("in sector")
is_review = data["tier"].eq("to review")
truth_all = data["relevance"].eq(1)
comparison_rows = []
volume_rows = []
for t in comparison_thresholds:
    cv_accept = data["p_cv"].ge(t)
    decisions = {
        "rule_a_plus_classifier": is_core | (is_review & cv_accept),
        "classifier_all": (is_core | is_review) & cv_accept,
    }
    for policy, decision in decisions.items():
        comparison_rows.append({"policy": policy, "threshold": t,
                                **binary_metrics(truth_all, decision)})
    production_accept = out["p_sector"].ge(t)
    production_core = out["tier"].eq("in sector")
    production_review = out["tier"].eq("to review")
    volume_rows.append({
        "threshold": t,
        "rule_a_plus_classifier": int((production_core | (production_review & production_accept)).sum()),
        "classifier_all": int(((production_core | production_review) & production_accept).sum()),
        "core_below_threshold": int((production_core & out["p_sector"].lt(t)).sum()),
        "core_without_prediction": int((production_core & out["p_sector"].isna()).sum()),
    })
policy_comparison = pd.DataFrame(comparison_rows)
policy_volumes = pd.DataFrame(volume_rows)
print("EXPLORATORY OOF comparison on the same scored labelled families")
print(policy_comparison.round(3).to_string(index=False))
print("\nFitted-model volumes in the scored output table (not accuracy estimates)")
print(policy_volumes.to_string(index=False))

# At the current threshold these are exactly the labelled decisions that change.
core_changes = shown[shown["tier"].eq("in sector") & shown["p_cv"].lt(threshold)].copy()
core_changes["effect"] = np.where(core_changes["relevance"].eq(1), "positive_lost", "negative_removed")
change_columns = ["docdb_family_id", "category", "p_cv", "effect", "files", "appln_title"]
print(f"\nRule A changes at threshold {threshold}:")
print(core_changes["effect"].value_counts().to_string())
print(core_changes[change_columns].sort_values("p_cv").to_string(index=False))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
policy_comparison.to_csv(OUTPUT_DIR / f"policy_comparison_{ENV}.csv", index=False)
policy_volumes.to_csv(OUTPUT_DIR / f"policy_volumes_{ENV}.csv", index=False)
core_changes[change_columns].to_csv(OUTPUT_DIR / f"rule_a_changes_{ENV}.csv", index=False)
